# Quantitative Analysis and Risk

Audience: Python learners preparing for quant interviews. Prerequisites: basic Python and pandas.

Goals: Returns, volatility, beta, exposure, skew, visualization.

Outline: setup, short tasks, checks, pitfalls. Run cells from top to bottom. All market data here is synthetic.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from market_simulator import (MarketMaker, ClientModel, MarketEnvironment,
    QuotedTrade, CompletedTrade, generate_market_data, generate_rfqs, run_simulation, make_quote)
from market_simulator.risk import beta, historical_volatility

# Resolve synthetic data from the installed local package, independent of kernel CWD.
import market_simulator
DATA = Path(market_simulator.__file__).resolve().parents[1] / 'data'
prices = pd.read_csv(DATA / 'market_prices.csv', parse_dates=['timestamp'])
requests = pd.read_csv(DATA / 'client_rfqs.csv', parse_dates=['timestamp'])
print(prices.head(2).to_string(index=False))


## Visual inspection
Plot one synthetic price path before calculating risk.

In [ ]:
prices.loc[prices.ticker == 'ALFA'].plot(x='timestamp', y='mid', title='Synthetic ALFA mid'); plt.show()

## Annualized volatility

Compute sample standard deviation of simple returns times sqrt(252). Require at least three positive prices. Example constant prices -> 0.


In [ ]:
def annualized_vol(prices_series: pd.Series) -> float:
    if len(prices_series) < 3 or prices_series.isna().any() or (prices_series <= 0).any():
        raise ValueError('need at least three positive prices')
    return float(prices_series.pct_change().dropna().std(ddof=1) * np.sqrt(252))

**Reasoning and complexity:** O(n) time; O(n) temporary space. State whether volatility is per day or per simulation step.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
assert annualized_vol(pd.Series([100,100,100])) == 0

## Dollar exposure

Return signed dollar exposure per ticker given signed positions and positive marks. Example {'A':-2}, {'A':100} -> {'A':-200}.


In [ ]:
def dollar_exposure(positions: dict[str, int], marks: dict[str, float]) -> dict[str, float]:
    return {t: q * marks[t] for t, q in positions.items()}

**Reasoning and complexity:** O(k) time and space. Preserve the sign; absolute exposure answers a different question.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
assert dollar_exposure({'A':-2}, {'A':100}) == {'A':-200}

## Inventory skew

Return (bid,ask) using `maker.quote`; a long position should move both sides below an otherwise identical flat quote. Explain why.


In [ ]:
def skewed_prices(maker: MarketMaker, ticker: str, mid: float) -> tuple[float, float]:
    q = maker.quote(ticker, 1, mid, 0)
    return q.bid_price, q.offer_price

**Reasoning and complexity:** O(1). A lower ask encourages a client buy, reducing maker inventory; a lower bid discourages further accumulation.

**Common mistake:** Forgetting the stated side, time ordering, or validation contract. An explicit loop is often a useful alternative when debugging.

In [ ]:
m = MarketMaker(inventory_limit=10)
flat = skewed_prices(m, 'A', 100)
m.add_trade(CompletedTrade('A', 5, 99, 'buy', 100, 99, 101, 0))
assert skewed_prices(m, 'A', 100)[0] < flat[0]

## Extension
Try a different random seed or add a transaction fee. Explain how your answer changes.